# **LLM Meal Planner**

In [1]:
import os
import sys
from datetime import datetime, timedelta

from openai import OpenAI

sys.path.insert(0, os.path.abspath(".."))

from engines import LLMMealPlanner, make_preferences, make_recipe
from models import (
    MealPlanningEnvironment,
    NutritionalInformation,
    Pantry,
    PantryIngredient,
)
from models.DietaryTag import DietaryTag


In [2]:
preferences = make_preferences()

In [3]:
recipes = [
    make_recipe(
        "Grilled Chicken & Rice",
        {"chicken breast": 200, "rice": 150, "olive oil": 10},
        calories=620,
        protein=52,
    ),
    make_recipe(
        "Vegetable Stir Fry",
        {"broccoli": 200, "carrot": 100, "soy sauce": 20, "rice": 150},
        calories=380,
        protein=12,
        tags=[DietaryTag.VEGAN, DietaryTag.VEGETARIAN],
    ),
    make_recipe(
        "Pasta Bolognese",
        {"pasta": 200, "ground beef": 150, "tomato": 100, "onions": 50},
        calories=710,
        protein=38,
    ),
    make_recipe(
        "Scrambled Eggs on Toast",
        {"eggs": 150, "butter": 15, "bread": 80},
        calories=420,
        protein=22,
        tags=[DietaryTag.VEGETARIAN],
    ),
    make_recipe(
        "Lentil Soup",
        {"dried brown lentils": 150, "carrot": 80, "celery": 60, "garlic": 10},
        calories=310,
        protein=20,
        tags=[DietaryTag.VEGAN, DietaryTag.VEGETARIAN, DietaryTag.GLUTEN_FREE, DietaryTag.LACTOSE_FREE],
    ),
]

print(f"{len(recipes)} recipes loaded:")
for i, r in enumerate(recipes):
    print(f"  {i}: {r.name}  ({r.nutritional_information.calories} kcal, {r.nutritional_information.protein}g protein)")


5 recipes loaded:
  0: Grilled Chicken & Rice  (620 kcal, 52g protein)
  1: Vegetable Stir Fry  (380 kcal, 12g protein)
  2: Pasta Bolognese  (710 kcal, 38g protein)
  3: Scrambled Eggs on Toast  (420 kcal, 22g protein)
  4: Lentil Soup  (310 kcal, 20g protein)


In [4]:
CURRENT_DATE = datetime.now()

pantry = Pantry()

pantry_items = [
    ("chicken breast", 800, 2.50, 3),  # expiring soon
    ("rice", 1000, 0.80, 14),
    ("broccoli", 400, 1.20, 5),
    ("eggs", 300, 0.30, 7),
]

for name, quantity, cost, days in pantry_items:
    pi = PantryIngredient(
        name=name,
        nutritional_information=NutritionalInformation(),
        estimated_expiration_date=CURRENT_DATE + timedelta(days=days),
        estimated_financial_cost=cost,
    )
    pantry.add(pi, quantity)

pantry.print()


---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-04-29
	Estimated Financial Cost per 100g: EUR 2.50
	Nutritional Information:
		Calories: None kcal
		Carbohydrates: None g
		Sugar: None g
		Protein: None g
		Fat: None g
		Saturated Fat: None g
		Fiber: None g
		Sodium: None mg
		Gluten Free: No
		Lactose Free: No
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-05-10
	Estimated Financial Cost per 100g: EUR 0.80
	Nutritional Information:
		Calories: None kcal
		Carbohydrates: None g
		Sugar: None g
		Protein: None g
		Fat: None g
		Saturated Fat: None g
		Fiber: None g
		Sodium: None mg
		Gluten Free: No
		Lactose Free: No
		Vegetarian: No
		Vegan: No
---
---
Quantity: 400 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-05-01
	Estimated Financial Cost per 100g: EUR 1.20
	Nutritional Information:
		Calories: None kcal
		Carbohydrates: None g
		Sugar: None g
		Protein: None g
		Fat: None g
		Satu

In [5]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=recipes,
    pantry=pantry,
    preferences=preferences,
)

print(f"{len(meal_planning_environment.recipes)} recipes available after dietary filtering")

5 recipes available after dietary filtering


In [6]:
client = OpenAI()  # reads OPENAI_API_KEY from environment

planner = LLMMealPlanner(meal_planning_environment, llm_client=client)

In [7]:
best_meal_plan, best_fitness_score = planner.generate_meal_plan()

In [8]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Lentil Soup,Lentil Soup,Lentil Soup
1,2,Lentil Soup,Lentil Soup,Lentil Soup
2,3,Lentil Soup,Lentil Soup,Lentil Soup
3,4,Lentil Soup,Lentil Soup,Lentil Soup
4,5,Lentil Soup,Lentil Soup,Lentil Soup
5,6,Lentil Soup,Lentil Soup,Lentil Soup
6,7,Lentil Soup,Lentil Soup,Lentil Soup


In [9]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,0,800,2d
1,rice,1000,0,1000,13d
2,broccoli,400,0,400,4d
3,eggs,300,0,300,6d


In [10]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: €{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 4
Total cost: €63.00


,Ingredient,Quantity to Buy (g),Cost (€)
0,carrot,1680.0,16.8
1,celery,1260.0,12.6
2,dried brown lentils,3150.0,31.5
3,garlic,210.0,2.1
4,TOTAL,,63.0


In [11]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,930 kcal,60 g,-1070.0 kcal,10.0 g
Day 2,930 kcal,60 g,-1070.0 kcal,10.0 g
Day 3,930 kcal,60 g,-1070.0 kcal,10.0 g
Day 4,930 kcal,60 g,-1070.0 kcal,10.0 g
Day 5,930 kcal,60 g,-1070.0 kcal,10.0 g
Day 6,930 kcal,60 g,-1070.0 kcal,10.0 g
Day 7,930 kcal,60 g,-1070.0 kcal,10.0 g
